In [ ]:
# aktuelle Implementierung als mapping
import cbrkit
import polars as pl
casebase = cbrkit.loaders.polars(pl.read_csv("data/cars-10m.csv"))
testcase = {"make": "max", "price": 10000, "miles": 20000}
sims = []
sim_num = cbrkit.sim.numbers.exponential(.01)
sim_text = cbrkit.sim.strings.levenshtein()
sim_global = cbrkit.sim.aggregator("mean")
cases = []
for idx, case in enumerate(casebase):
    sims.append((sim_global([sim_num(case[0], testcase["price"]), sim_num(case[5], testcase["miles"]), sim_text(case[3], testcase["make"])]), idx))
    cases.append(case)
res = sorted(sims, key=lambda x: x[0], reverse=True)
for idx in range(20):
    print(cases[res[idx][1]])

(9999, 2013, 'ford', 'b-max', 'petrol', 20000, 'clean', 'manual', '4wd', 'other', 'black')
(9999, 2013, 'ford', 'b-max', 'petrol', 20000, 'clean', 'manual', '4wd', 'other', 'black')
(9999, 2013, 'ford', 'b-max', 'petrol', 20000, 'clean', 'manual', '4wd', 'other', 'black')
(9999, 2013, 'ford', 'b-max', 'petrol', 20000, 'clean', 'manual', '4wd', 'other', 'black')
(9999, 2013, 'ford', 'b-max', 'petrol', 20000, 'clean', 'manual', '4wd', 'other', 'black')
(9999, 2013, 'ford', 'b-max', 'petrol', 20000, 'clean', 'manual', '4wd', 'other', 'black')
(9998, 2015, 'porsche', 'panamera', 'petrol', 20000, 'rebuilt', 'manual', 'fwd', 'other', 'black')
(9998, 2015, 'porsche', 'panamera', 'petrol', 20000, 'rebuilt', 'manual', 'fwd', 'other', 'black')
(9998, 2015, 'porsche', 'panamera', 'petrol', 20000, 'rebuilt', 'manual', 'fwd', 'other', 'black')
(9998, 2015, 'porsche', 'panamera', 'petrol', 20000, 'rebuilt', 'manual', 'fwd', 'other', 'black')
(9998, 2015, 'porsche', 'panamera', 'petrol', 20000, 'rebu

In [6]:
# polars - eager api
import Levenshtein
df = pl.read_csv("data/cars-10m.csv")
testcase = {"make": "max", "price": 10000, "miles": 20000}
neg_alpha = -0.01

df = df.with_columns(
    (pl.col("price").sub(testcase["price"]).abs().mul(neg_alpha).exp().alias("price_sim"), 
    pl.col("miles").sub(testcase["miles"]).abs().mul(neg_alpha).exp().alias("miles_sim"),
     pl.col("make").map_elements(lambda x: Levenshtein.ratio(x, testcase["make"]), return_dtype=pl.Float32).alias("make_sim"))
)
df = df.with_columns(
    pl.col("price_sim").add(pl.col("miles_sim")).add(pl.col("make_sim")).mul(.333).alias("global_sim")
).sort("global_sim",descending=True)
df.head(20)


price,year,manufacturer,make,fuel,miles,title_status,transmission,drive,type,paint_color,price_sim,miles_sim,make_sim,global_sim
i64,i64,str,str,str,i64,str,str,str,str,str,f64,f64,f32,f64
9999,2013,"""ford""","""b-max""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.99005,1.0,0.75,0.912437
9999,2013,"""ford""","""b-max""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.99005,1.0,0.75,0.912437
9999,2013,"""ford""","""b-max""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.99005,1.0,0.75,0.912437
9999,2013,"""ford""","""b-max""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.99005,1.0,0.75,0.912437
9999,2013,"""ford""","""b-max""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.99005,1.0,0.75,0.912437
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
10008,2011,"""audi""","""a5""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.923116,1.0,0.4,0.773598
10008,2011,"""audi""","""a5""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.923116,1.0,0.4,0.773598
10008,2011,"""audi""","""a5""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.923116,1.0,0.4,0.773598


In [7]:
# polars - lazy api
import Levenshtein
df = pl.scan_csv("data/cars-10m.csv")
testcase = {"make": "max", "price": 10000, "miles": 20000}
neg_alpha = -0.01

df = df.with_columns(
    (pl.col("price").sub(testcase["price"]).abs().mul(neg_alpha).exp().alias("price_sim"), 
    pl.col("miles").sub(testcase["miles"]).abs().mul(neg_alpha).exp().alias("miles_sim"),
     pl.col("make").map_elements(lambda x: Levenshtein.ratio(x, testcase["make"]), return_dtype=pl.Float32).alias("make_sim"))
)
df = df.with_columns(
    pl.col("price_sim").add(pl.col("miles_sim")).add(pl.col("make_sim")).mul(.333).alias("global_sim")
).sort("global_sim",descending=True)
df.collect().head(20)


price,year,manufacturer,make,fuel,miles,title_status,transmission,drive,type,paint_color,price_sim,miles_sim,make_sim,global_sim
i64,i64,str,str,str,i64,str,str,str,str,str,f64,f64,f32,f64
9999,2013,"""ford""","""b-max""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.99005,1.0,0.75,0.912437
9999,2013,"""ford""","""b-max""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.99005,1.0,0.75,0.912437
9999,2013,"""ford""","""b-max""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.99005,1.0,0.75,0.912437
9999,2013,"""ford""","""b-max""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.99005,1.0,0.75,0.912437
9999,2013,"""ford""","""b-max""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.99005,1.0,0.75,0.912437
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
10008,2011,"""audi""","""a5""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.923116,1.0,0.4,0.773598
10008,2011,"""audi""","""a5""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.923116,1.0,0.4,0.773598
10008,2011,"""audi""","""a5""","""petrol""",20000,"""clean""","""manual""","""4wd""","""other""","""black""",0.923116,1.0,0.4,0.773598


In [ ]:
# pandas
import pandas as pd
import numpy as np
import Levenshtein

df = pd.read_csv("data/cars-10m.csv")
testcase = {"make": "max", "price": 10000, "miles": 20000}
neg_alpha = -0.01

df["price_sim"] = (df["price"] - testcase["price"]).abs() * neg_alpha
df["price_sim"] = df["price_sim"].apply(lambda x: np.exp(x))

df["miles_sim"] = (df["miles"] - testcase["miles"]).abs() * neg_alpha
df["miles_sim"] = df["miles_sim"].apply(lambda x: np.exp(x))

df["make_sim"] = df["make"].apply(lambda x: Levenshtein.ratio(x, testcase["make"]))

df["global_sim"] = (df["price_sim"] + df["miles_sim"] + df["make_sim"]) * 0.333
df = df.sort_values(by="global_sim", ascending=False)

df.head(20)


,price,year,manufacturer,make,fuel,miles,title_status,transmission,drive,type,paint_color,price_sim,miles_sim,make_sim,global_sim
10098957,9999,2013,ford,b-max,petrol,20000,clean,manual,4wd,other,black,0.990050,1.0,0.750000,0.912437
985557,9999,2013,ford,b-max,petrol,20000,clean,manual,4wd,other,black,0.990050,1.0,0.750000,0.912437
2808237,9999,2013,ford,b-max,petrol,20000,clean,manual,4wd,other,black,0.990050,1.0,0.750000,0.912437
4630917,9999,2013,ford,b-max,petrol,20000,clean,manual,4wd,other,black,0.990050,1.0,0.750000,0.912437
6453597,9999,2013,ford,b-max,petrol,20000,clean,manual,4wd,other,black,0.990050,1.0,0.750000,0.912437
8276277,9999,2013,ford,b-max,petrol,20000,clean,manual,4wd,other,black,0.990050,1.0,0.750000,0.912437
2402722,9998,2015,porsche,panamera,petrol,20000,rebuilt,manual,fwd,other,black,0.980199,1.0,0.363636,0.780497
9693442,9998,2015,porsche,panamera,petrol,20000,rebuilt,manual,fwd,other,black,0.980199,1.0,0.363636,0.780497
4225402,9998,2015,porsche,panamera,petrol,20000,rebuilt,manual,fwd,other,black,0.980199,1.0,0.363636,0.780497
580042,9998,2015,porsche,panamera,petrol,20000,rebuilt,manual,fwd,other,black,0.980199,1.0,0.363636,0.780497
